# 09 — P2: Train Standard ViT-Tiny on MNIST

**Plan 2 — Phase 1.1 / 1.2**: Standard cross-entropy training of the ViT-Tiny architecture used
throughout the compositional MILP+Lipschitz verification pipeline.

## Architecture (fixed across Plan 2)
| Param | Value |
|---|---|
| img_size | 28 |
| patch_size | 4  → 49 tokens |
| embed_dim | 64 |
| num_heads | 2  (head_dim 32) |
| num_layers | **2** (kept small for tractable MILP) |
| mlp_ratio | 2  (hidden=128) |
| activation | ReLU |
| norm | **RMSNorm** (pre-LN) |
| pooling | mean over tokens (no CLS) |
| pos embed | learnable additive |
| Q/K bias | False |

## Output
Checkpoint at `runs/vit_tiny_standard/model.pt`. Target ≥97% test accuracy.

In [1]:
# ── 1. Install (Colab-friendly; harmless on local) ────────────────────────
!pip install -q numpy pandas torch torchvision tqdm pyyaml

In [2]:
# ── 2. Imports & reproducibility ──────────────────────────────────────────
from __future__ import annotations
import math, json, time, os, warnings
from copy import deepcopy
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

def set_seed(seed: int = 1234) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(1234)
print(f'Device : {device}')
print(f'PyTorch: {torch.__version__}')

Device : cuda
PyTorch: 2.10.0+cu128


In [3]:
# ── 3. Config ─────────────────────────────────────────────────────────────
CFG = dict(
    seed         = 1234,
    data_root    = '/tmp/mnist',
    run_dir      = 'runs/vit_tiny_standard',
    n_epochs     = 15,
    batch_size   = 128,
    lr           = 3e-4,
    weight_decay = 0.01,
    # Architecture
    img_size     = 28,
    patch_size   = 4,
    embed_dim    = 64,
    num_heads    = 2,
    num_layers   = 2,
    mlp_ratio    = 2,
    eps_rms      = 1e-6,
)
Path(CFG['run_dir']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2))

{
  "seed": 1234,
  "data_root": "/tmp/mnist",
  "run_dir": "runs/vit_tiny_standard",
  "n_epochs": 15,
  "batch_size": 128,
  "lr": 0.0003,
  "weight_decay": 0.01,
  "img_size": 28,
  "patch_size": 4,
  "embed_dim": 64,
  "num_heads": 2,
  "num_layers": 2,
  "mlp_ratio": 2,
  "eps_rms": 1e-06
}


In [4]:
# ── 4. Data loaders ───────────────────────────────────────────────────────
def get_mnist_loaders(batch_size=128, data_root='/tmp/mnist'):
    tf = transforms.ToTensor()
    train_ds = torchvision.datasets.MNIST(data_root, train=True,  download=True, transform=tf)
    test_ds  = torchvision.datasets.MNIST(data_root, train=False, download=True, transform=tf)
    kw = dict(num_workers=0, pin_memory=False)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kw)
    test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)
    return train_loader, test_loader

train_loader, test_loader = get_mnist_loaders(CFG['batch_size'], CFG['data_root'])
print(f'Train: {len(train_loader.dataset):,}  Test: {len(test_loader.dataset):,}')

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.52MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 133kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.02MB/s]

Train: 60,000  Test: 10,000


In [5]:
# ── 5. ViT-Tiny model definition ──────────────────────────────────────────
#
# Differences from a textbook ViT (intentional, all chosen for MILP-friendliness):
#   • RMSNorm instead of LayerNorm   — no mean subtraction → simpler MILP
#   • Mean-pool over tokens          — no CLS token bookkeeping in MILP
#   • ReLU                            — piecewise-linear, big-M MILP
#   • Q/K projections without bias    — symmetry simplifies attention encoding
#
# Implementation note: every sub-module is exposed as a plain attribute so
# downstream phases (Lipschitz / IBP / MILP) can iterate cleanly.

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.dim    = dim
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)


class PatchEmbed(nn.Module):
    """Non-overlapping patch projection: stride==kernel==patch_size."""
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size   = img_size
        self.patch_size = patch_size
        self.n_patches  = (img_size // patch_size) ** 2  # 49 for 28/4
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)                       # (B, E, H/P, W/P)
        return x.flatten(2).transpose(1, 2)    # (B, n_patches, E)


class MHSA(nn.Module):
    """Multi-head self-attention. No bias on Q/K (per Plan 2 spec)."""
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        H, D    = self.num_heads, self.head_dim
        q = self.W_q(x).view(B, N, H, D).transpose(1, 2)  # (B, H, N, D)
        k = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        v = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # (B,H,N,N)
        attn   = F.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v)                              # (B,H,N,D)
        out    = out.transpose(1, 2).contiguous().view(B, N, C)     # (B,N,E)
        return self.W_o(out)


class MLPBlock(nn.Module):
    def __init__(self, embed_dim: int, mlp_ratio: int = 2):
        super().__init__()
        hidden   = embed_dim * mlp_ratio
        self.fc1 = nn.Linear(embed_dim, hidden)
        self.fc2 = nn.Linear(hidden, embed_dim)
        self.act = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    """Pre-LN: x = x + MHSA(RMSNorm(x));  x = x + MLP(RMSNorm(x))."""
    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: int = 2,
                 eps_rms: float = 1e-6):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps_rms)
        self.attn  = MHSA(embed_dim, num_heads)
        self.norm2 = RMSNorm(embed_dim, eps_rms)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ViTTiny(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, num_classes=10,
                 embed_dim=64, num_heads=2, num_layers=2, mlp_ratio=2,
                 eps_rms=1e-6):
        super().__init__()
        self.cfg = dict(img_size=img_size, patch_size=patch_size,
                        in_channels=in_channels, num_classes=num_classes,
                        embed_dim=embed_dim, num_heads=num_heads,
                        num_layers=num_layers, mlp_ratio=mlp_ratio,
                        eps_rms=eps_rms)
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.patch_embed.n_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, eps_rms)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(embed_dim, eps_rms)
        self.head = nn.Linear(embed_dim, num_classes)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Conv2d):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)               # (B, N, E)
        x = x + self.pos_embed                # learnable additive position
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        x = x.mean(dim=1)                     # mean-pool over tokens
        return self.head(x)

# Sanity check
with torch.no_grad():
    _m = ViTTiny(**{k: CFG[k] for k in ('img_size','patch_size','embed_dim',
                                        'num_heads','num_layers','mlp_ratio','eps_rms')})
    _x = torch.zeros(2, 1, CFG['img_size'], CFG['img_size'])
    _y = _m(_x)
    n_params = sum(p.numel() for p in _m.parameters())
print(f'ViT-Tiny  params={n_params:,}  output={tuple(_y.shape)}')

ViT-Tiny  params=71,370  output=(2, 10)


In [6]:
# ── 6. Training utilities ─────────────────────────────────────────────────
def train_epoch(model, loader, opt, device):
    model.train()
    tot_loss = correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        opt.step()
        with torch.no_grad():
            tot_loss += loss.item() * len(y)
            correct  += (logits.argmax(1) == y).sum().item()
            total    += len(y)
    return tot_loss / total, correct / total

@torch.no_grad()
def eval_acc(model, loader, device) -> float:
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += len(y)
    return correct / total

In [7]:
# ── 7. Train ──────────────────────────────────────────────────────────────
set_seed(CFG['seed'])
model = ViTTiny(
    img_size=CFG['img_size'], patch_size=CFG['patch_size'],
    embed_dim=CFG['embed_dim'], num_heads=CFG['num_heads'],
    num_layers=CFG['num_layers'], mlp_ratio=CFG['mlp_ratio'],
    eps_rms=CFG['eps_rms'],
).to(device)

opt   = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['n_epochs'])

best_acc, best_state = 0.0, None
history = []
for ep in range(1, CFG['n_epochs'] + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, opt, device)
    te_acc          = eval_acc(model, test_loader, device)
    sched.step()
    history.append(dict(epoch=ep, train_loss=tr_loss, train_acc=tr_acc,
                        test_acc=te_acc, lr=opt.param_groups[0]['lr']))
    if te_acc > best_acc:
        best_acc, best_state = te_acc, deepcopy(model.state_dict())
    print(f'ep {ep:02d}/{CFG["n_epochs"]:02d}  '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  '
          f'test_acc={te_acc:.4f}  ({time.time()-t0:.0f}s)')

model.load_state_dict(best_state)
print(f'\nBest test accuracy: {best_acc:.4f}')
print(f'Target ≥ 0.97 — {"PASS" if best_acc >= 0.97 else "BELOW TARGET (still usable)"}')

ep 01/15  train_loss=1.0272  train_acc=0.6690  test_acc=0.8825  (12s)
ep 02/15  train_loss=0.3288  train_acc=0.9017  test_acc=0.9217  (11s)
ep 03/15  train_loss=0.2245  train_acc=0.9311  test_acc=0.9358  (11s)
ep 04/15  train_loss=0.1740  train_acc=0.9478  test_acc=0.9390  (11s)
ep 05/15  train_loss=0.1428  train_acc=0.9566  test_acc=0.9570  (12s)
ep 06/15  train_loss=0.1225  train_acc=0.9624  test_acc=0.9565  (11s)
ep 07/15  train_loss=0.1064  train_acc=0.9675  test_acc=0.9674  (12s)
ep 08/15  train_loss=0.0942  train_acc=0.9712  test_acc=0.9656  (11s)
ep 09/15  train_loss=0.0831  train_acc=0.9745  test_acc=0.9706  (11s)
ep 10/15  train_loss=0.0707  train_acc=0.9785  test_acc=0.9744  (11s)
ep 11/15  train_loss=0.0604  train_acc=0.9815  test_acc=0.9754  (11s)
ep 12/15  train_loss=0.0523  train_acc=0.9843  test_acc=0.9768  (12s)
ep 13/15  train_loss=0.0460  train_acc=0.9866  test_acc=0.9782  (11s)
ep 14/15  train_loss=0.0407  train_acc=0.9886  test_acc=0.9781  (11s)
ep 15/15  train_loss

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

drive_path = Path('/content/drive/My Drive/formal-verification/runs/vit_tiny_standard')
drive_path.mkdir(parents=True, exist_ok=True)
ckpt_path = drive_path / 'model.pt'
torch.save({
    'state_dict': model.state_dict(),
    'cfg':        model.cfg,
    'best_acc':   best_acc,
    'history':    history,
    'training':   {k: CFG[k] for k in ('seed','n_epochs','batch_size','lr','weight_decay')},
}, ckpt_path)
print(f'Saved → {ckpt_path}  ({ckpt_path.stat().st_size/1e3:.1f} KB)')
print(f'Best test acc: {best_acc:.4f}')

In [8]:
# ── 8. Save checkpoint ────────────────────────────────────────────────────
ckpt_path = Path(CFG['run_dir']) / 'model.pt'
torch.save({
    'state_dict': model.state_dict(),
    'cfg':        model.cfg,
    'best_acc':   best_acc,
    'history':    history,
    'training':   {k: CFG[k] for k in ('seed','n_epochs','batch_size','lr','weight_decay')},
}, ckpt_path)
print(f'Saved → {ckpt_path}  ({ckpt_path.stat().st_size/1e3:.1f} KB)')
print(f'Best test acc: {best_acc:.4f}')

Saved → runs/vit_tiny_standard/model.pt  (296.0 KB)
Best test acc: 0.9787


In [9]:
cd runs/vit_tiny_standard/

/content/runs/vit_tiny_standard


In [10]:
cd ../

/content/runs


In [11]:
ls

vit_tiny_standard/


In [12]:
cd ../

/content


In [13]:
ls

runs/  sample_data/
